# 🚀 Master Notebook: Huấn Luyện AI 2.0 Multi-Agent RAG Trên Kaggle Dual T4
### Tích hợp công thức SOTA từ Unsloth (Qwen-Embedding-4B), MoAI, CoREB và Turkish Legal RAG
- **Phần cứng**: Kaggle Dual Tesla T4 (`2 x 16GB = 32GB VRAM`)
- **Tối ưu**: `CachedMultipleNegativesRankingLoss` (GradCache) + LoRA FP16
- **Tiêu chuẩn**: Đánh giá thực nghiệm độc lập bằng `Recall@10`, `nDCG@10`, `MRR@10` (Ablation Study A1 -> A4)

In [1]:
def patch_environment():
    """Patches missing triton.ops, removes invalid BNB_CUDA_VERSION overrides, and handles CPU/GPU fallbacks for Kaggle and Colab runtimes."""
    import os
    import sys
    import types

    # 1. Clean BNB_CUDA_VERSION if it causes missing libcudart.so
    if "BNB_CUDA_VERSION" in os.environ:
        del os.environ["BNB_CUDA_VERSION"]

    # 2. Patch Triton ops mock for Triton 3+ compatibility
    m_triton = sys.modules.get("triton")
    if m_triton is None:
        try:
            import triton
            m_triton = triton
        except Exception:
            m_triton = types.ModuleType("triton")
            m_triton.__path__ = []
            sys.modules["triton"] = m_triton

    m_ops = types.ModuleType("triton.ops")
    m_ops.__path__ = []
    m_perf = types.ModuleType("triton.ops.matmul_perf_model")
    m_perf.early_config_prune = lambda *a, **k: None
    m_perf.estimate_matmul_time = lambda *a, **k: 0
    m_ops.matmul_perf_model = m_perf
    setattr(m_triton, "ops", m_ops)
    sys.modules["triton.ops"] = m_ops
    sys.modules["triton.ops.matmul_perf_model"] = m_perf

    # 3. Bulletproof fallback for bitsandbytes if native CUDA libs fail or not present
    try:
        import bitsandbytes as bnb
    except Exception:
        import torch
        m_bnb = types.ModuleType("bitsandbytes")
        m_bnb_nn = types.ModuleType("bitsandbytes.nn")

        class DummyBnbLinear(torch.nn.Linear):
            pass

        m_bnb_nn.Linear8bitLt = DummyBnbLinear
        m_bnb_nn.Linear4bit = DummyBnbLinear
        m_bnb_nn.modules = types.ModuleType("bitsandbytes.nn.modules")
        m_bnb.nn = m_bnb_nn
        m_bnb.__version__ = "0.44.1"
        sys.modules["bitsandbytes"] = m_bnb
        sys.modules["bitsandbytes.nn"] = m_bnb_nn
        sys.modules["bitsandbytes.nn.modules"] = m_bnb_nn.modules

patch_environment()

# Kiểm tra phần cứng và thư viện
import torch
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device Count: {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  - GPU {i}: {torch.cuda.get_device_name(i)}')


CUDA Available: True
Device Count: 2
  - GPU 0: Tesla T4
  - GPU 1: Tesla T4


In [3]:
# BƯỚC 2: CÀI ĐẶT THƯ VIỆN ĐÃ ĐƯỢC CỐ ĐỊNH PHIÊN BẢN (PINNED SOTA STACK)
!pip install -q \
    transformers==4.46.2 \
    peft==0.13.2 \
    trl==0.12.1 \
    sentence-transformers==3.2.1 \
    bitsandbytes \
    accelerate==1.0.1 \
    datasets==3.0.2 \
    safetensors==0.4.5 \
    ninja PyMuPDF scikit-learn scipy einops huggingface_hub

print('✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!')


✅ Đã cài đặt xong toàn bộ thư viện SOTA tương thích 100% với Tesla T4!


In [50]:
# BƯỚC 3: HUẤN LUYỆN CROSS-ENCODER RERANKER (RECIPE CoREB: LoRA r=16, alpha=16, nDCG@10)
import os, sys, urllib.request

# 1. Tự động tải đè bản vá train_reranker_lora.py mới nhất từ GitHub
dest_path = "/kaggle/working/training_lab/training/reranker/train_reranker_lora.py" if os.path.exists("/kaggle/working/training_lab") else "training_lab/training/reranker/train_reranker_lora.py"
raw_url = "https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/main/training_lab/training/reranker/train_reranker_lora.py"

try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    urllib.request.urlretrieve(raw_url, dest_path)
    print("✅ Đã cập nhật thành công train_reranker_lora.py bản vá mới nhất!")
except Exception as e:
    print(f"⚠️ Lỗi tải file: {e}")

# 2. Chạy huấn luyện với chính xác Python có Dual GPU T4
py_bin = sys.executable
print('\n' + '='*80 + '\n🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES\n' + '='*80)
mod = "training_lab.training.reranker.train_reranker_lora" if os.path.exists("training_lab") else "training.reranker.train_reranker_lora"
dpath = "training_lab/data/triplets/massive_reranker_pairs.jsonl" if os.path.exists("training_lab") else "data/triplets/massive_reranker_pairs.jsonl"

!{py_bin} -m {mod} --epochs 3 --lr 3e-5 --data {dpath}

✅ Đã cập nhật thành công train_reranker_lora.py bản vá mới nhất!

🎯 [STAGE 1] HUẤN LUYỆN RERANKER CROSS-ENCODER VỚI HARD NEGATIVES
I0000 00:00:1794068673.586552     796 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
🚀 [CoREB Reranker] Initializing Cross-Encoder: BAAI/bge-reranker-v2-m3 on available GPUs...
trainable params: 8,161,281 || all params: 575,949,826 || trainable%: 1.4170
Map: 100%|█████████████████████████| 3000/3000 [00:01<00:00, 2690.65 examples/s]
📊 [Reranker DataLoader] Train: 17002, Validation: 3000
⚡ Starting LoRA Reranker Training with MoAI Checkpoint Selection (nDCG@10)...
  0%|                                                  | 0/1593 [00:00<?, ?it/s]/usr/local/lib/python3.13/dist-packages/torch/autograd/function.py:596: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
{'loss': 41.1786

In [53]:
import os
from IPython.display import FileLink

# Chuyển về đúng thư mục working và truyền đường dẫn tương đối (không lặp lại /kaggle/working)
os.chdir('/kaggle/working')
FileLink('Reranker_LoRA_Best_DualT4.zip')

/kaggle/working/Reranker_LoRA_Best_DualT4.zip

In [7]:
# ==============================================================================
# BƯỚC 4: HUẤN LUYỆN EMBEDDING (RECIPE MoAI: 1 EPOCH SOTA, TỰ ĐỘNG NẠP DỮ LIỆU)
# ==============================================================================
import os, sys, time, urllib.request

# 0. Tự động kiểm tra và nạp bộ dữ liệu nếu phiên Kaggle vừa khởi động lại
dpath = None
for cand in [
    "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "/kaggle/working/Multi-Agent_RAG/training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "training_lab/data/triplets/massive_embedding_triplets.jsonl",
    "data/triplets/massive_embedding_triplets.jsonl"
]:
    if os.path.exists(cand):
        dpath = cand
        break

if not dpath:
    print("⏳ Phiên Kaggle mới chưa có dữ liệu, đang tự động clone repository từ GitHub...", flush=True)
    !git clone --depth 1 https://github.com/ducmanh-jr/Multi-Agent_RAG.git /kaggle/working/temp_repo
    !cp -rn /kaggle/working/temp_repo/training_lab /kaggle/working/
    !rm -rf /kaggle/working/temp_repo
    dpath = "/kaggle/working/training_lab/data/triplets/massive_embedding_triplets.jsonl"
    print("✅ Đã nạp thành công bộ dữ liệu massive_embedding_triplets.jsonl!")

# 1. Cập nhật thư viện tương thích tuyệt đối
!pip install -q 'accelerate>=1.1.0' --upgrade sentence-transformers

# 2. Tải bản vá triệt tiêu 100% lỗi import bằng URL Commit Hash trực tiếp
dest_path = "/kaggle/working/training_lab/training/embedding/train_embedding_lora.py" if os.path.exists("/kaggle/working/training_lab") else "training_lab/training/embedding/train_embedding_lora.py"
raw_url = "https://raw.githubusercontent.com/ducmanh-jr/Multi-Agent_RAG/101915a/training_lab/training/embedding/train_embedding_lora.py"

try:
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    req = urllib.request.Request(raw_url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as resp, open(dest_path, 'wb') as out_f:
        out_f.write(resp.read())
    print("✅ Đã cập nhật thành công train_embedding_lora.py bản vá mới nhất!")
except Exception as e:
    print(f"ℹ️ Dùng file local ({e})")

# 3. Khởi chạy 1 Epoch chuẩn (~625 steps, hoàn tất trong ~2.5 tiếng)
py_bin = sys.executable
print('\n' + '='*80 + '\n⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING 1 EPOCH CHUẨN VÀNG SOTA (~2.5 TIẾNG)\n' + '='*80)
if os.path.exists("/kaggle/working/training_lab"):
    os.chdir("/kaggle/working")
    mod = "training_lab.training.embedding.train_embedding_lora"
elif os.path.exists("training_lab"):
    mod = "training_lab.training.embedding.train_embedding_lora"
else:
    mod = "training.embedding.train_embedding_lora"

!{py_bin} -m {mod} --epochs 1 --lr 3e-5 --data {dpath} --batch-size 8

⏳ Phiên Kaggle mới chưa có dữ liệu, đang tự động clone repository từ GitHub...
Cloning into '/kaggle/working/temp_repo'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (554/554), done.
remote: Compressing objects: 100% (500/500), done.
remote: Total 554 (delta 61), reused 425 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (554/554), 264.94 MiB | 31.96 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (490/490), done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead
✅ Đã nạp thành công bộ dữ liệu massive_embedding_triplets.jsonl!
✅ Đã cập nhật thành công train_embedding_lora.py bản vá mới nhất!

⚡ [STAGE 2] HUẤN LUYỆN EMBEDDING 1 EPOCH CHUẨN VÀNG SOTA (~2.5 TIẾNG)
/kaggle/working/training_lab/training/embedding/train_embedding_lora.py:26: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_tra

In [8]:
import os, shutil
from pathlib import Path
from IPython.display import FileLink

os.chdir('/kaggle/working')

# 1. Tìm thư mục lưu trọng số Embedding LoRA vừa huấn luyện
embed_dir = None
for cand in [
    Path('/kaggle/working/training_lab/training/embedding/weights'),
    Path('/kaggle/working/training_lab/models/adapters/embedding_lora'),
    Path('training_lab/training/embedding/weights')
]:
    if cand.exists():
        embed_dir = cand
        break

# 2. Đóng gói thành file zip
zip_name = "Embedding_LoRA_Best_DualT4.zip"
if embed_dir:
    shutil.make_archive(zip_name.replace(".zip", ""), "zip", embed_dir)
    print(f"🎉 Đã đóng gói thành công: {zip_name}")

# 3. Tạo link click tải về máy tính 1-Click
FileLink(zip_name)

🎉 Đã đóng gói thành công: Embedding_LoRA_Best_DualT4.zip


/kaggle/working/Embedding_LoRA_Best_DualT4.zip

In [ ]:
import os, sys, shutil
from pathlib import Path
from IPython.display import FileLink

print("=" * 80)
print("📝 [BẢN VÁ 2d17fae] HUẤN LUYỆN AGENT 5 REPORT WRITER TRÊN KAGGLE DUAL T4")
print("=" * 80)

# 1. Kéo bản vá llm_blender mới nhất từ GitHub
!rm -rf /kaggle/working/temp_sync
!git clone --depth 1 https://github.com/ducmanh-jr/Multi-Agent_RAG.git /kaggle/working/temp_sync
!cp -rf /kaggle/working/temp_sync/training_lab/agent5_writer /kaggle/working/training_lab/
!rm -rf /kaggle/working/temp_sync
print("✅ Đã cập nhật xong mã nguồn Agent 5 mới nhất!")

# 2. Khởi chạy huấn luyện Agent 5 trên 2 GPU Dual T4 (~40-45 phút)
target_script = "/kaggle/working/training_lab/agent5_writer/train_agent5_t4x2.py"
!python {target_script} --epochs-sft 3 --epochs-dpo 2

# 3. Dọn dẹp checkpoints phụ để file zip tinh gọn (< 80MB)
!rm -rf /kaggle/working/training_lab/agent5_writer/weights/*checkpoints*
!rm -rf /kaggle/working/training_lab/agent5_writer/weights/**/checkpoints*

# 4. Đóng gói trọng số Agent 5 mới tinh vừa huấn luyện
os.chdir('/kaggle/working')
zip_out = "Agent5_Academic_Writer_DualT4.zip"
weights_dir = Path("/kaggle/working/training_lab/agent5_writer/weights")
shutil.make_archive(zip_out.replace(".zip", ""), "zip", weights_dir)
print(f"\n🎉 Đã hoàn thành huấn luyện và đóng gói: {zip_out}")

# 5. Hiện link click tải về máy tính 1-Click
FileLink(zip_out)

📝 [BẢN VÁ 2d17fae] HUẤN LUYỆN AGENT 5 REPORT WRITER TRÊN KAGGLE DUAL T4
Cloning into '/kaggle/working/temp_sync'...
remote: Enumerating objects: 554, done.
remote: Counting objects: 100% (554/554), done.
remote: Compressing objects: 100% (502/502), done.
remote: Total 554 (delta 61), reused 419 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (554/554), 264.95 MiB | 34.25 MiB/s, done.
Resolving deltas: 100% (61/61), done.
Updating files: 100% (490/490), done.
✅ Đã cập nhật xong mã nguồn Agent 5 mới nhất!
config.json: 100%|█████████████████████████████| 663/663 [00:00<00:00, 3.24MB/s]
tokenizer_config.json: 100%|███████████████| 7.30k/7.30k [00:00<00:00, 18.8MB/s]
vocab.json: 100%|██████████████████████████| 2.78M/2.78M [00:00<00:00, 75.4MB/s]
merges.txt: 100%|███████████████████████████| 1.67M/1.67M [00:00<00:00, 121MB/s]
tokenizer.json: 100%|███████████████████████| 7.03M/7.03M [00:00<00:00, 147MB/s]
⚡ [ModelBuilder] Balancing model layers across 2 GPUs with memory bounds: {